In [6]:
import gc
import torch
import cv2
from ultralytics import YOLO
import sqlite3
import os
import time
from datetime import datetime

In [7]:
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB")

0.0 MB


In [8]:
save_folder = "saved_image"
os.makedirs(save_folder, exist_ok=True)

db_path = "detection_history.db"
conn  = sqlite3.connect(db_path)
cursor = conn.cursor()

In [9]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS knife_alerts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        image_path TEXT,
        confidence REAL
    )
""")
conn.commit()

In [10]:
best_model = YOLO(r"C:\Ml-itsc\MoneyApp\Yolo_knife\runs\detect\train-6\weights\best.pt")

In [ ]:
cap = cv2.VideoCapture(0)
last_save_time = 0
coldown = 3.0

while True:

    ret, frame = cap.read()
    results = best_model(frame)

    annotated = results[0].plot()

    for result in results:
        for box in result.boxes:
            conf = float(box.conf)
            print(conf)

            if conf > 0.6:
                print("WARNING: KNIFE DETECTED")
                cv2.putText(
                    img=annotated,
                    text="WARNING: KNIFE DETECTED",
                    org=(50, 80),                    
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1.0,                   
                    color=(0, 0, 255),               
                    thickness=3,                     
                    lineType=cv2.LINE_AA             
                )

                curr_time = time.time()
                if curr_time - last_save_time >coldown:

                    now = datetime.now()
                    time_str = now.strftime("%Y%m%d_%H%M%S_%f") 
                    image_name = f"knife_{time_str}.jpg"
                    image_path = os.path.join(save_folder, image_name)

                    cv2.imwrite(image_path, annotated)

                    formatted_time = now.strftime("%Y-%m-%d %H:%M:%S")
                    cursor.execute("""
                        INSERT INTO knife_alerts (timestamp, image_path, confidence)
                        VALUES (?, ?, ?)
                    """, (formatted_time, image_path, conf))

                    conn.commit()

                    last_save_time = curr_time



    cv2.imshow("Knife Detector", annotated)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 (no detections), 73.1ms
Speed: 29.8ms preprocess, 73.1ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 12.8ms
Speed: 4.6ms preprocess, 12.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 15.0ms
Speed: 1.7ms preprocess, 15.0ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 15.3ms
Speed: 1.7ms preprocess, 15.3ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.9ms
Speed: 3.3ms preprocess, 11.9ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 14.2ms
Speed: 1.2ms preprocess, 14.2ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 12.5ms
Speed: 1.1ms preprocess, 12.5ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 13.5ms
Speed: 1.3ms preprocess, 13.5ms 